# Question Answering System — Model Walkthrough

This notebook demonstrates the CNN+BiLSTM+Attention QA model architecture,
training procedure, and inference.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

from model.qa_model import ExtractiveQAModel, QAModelConfig, count_parameters, decode_best_span
from preprocessing.vocabulary import Vocabulary
from preprocessing.tokenizer import tokenize_with_offsets
import torch

## 1. Load the trained model

The model checkpoint was saved during training to `saved_models/`.

In [ ]:
config = QAModelConfig.load("../saved_models/model_config.json")
vocab  = Vocabulary.load("../saved_models/vocab.json")

model = ExtractiveQAModel(config)
state = torch.load("../saved_models/qa_model.pt", map_location="cpu", weights_only=True)
if "state_dict" in state:
    state = state["state_dict"]
model.load_state_dict(state)
model.eval()

print(f"Parameters: {count_parameters(model):,}")
print(f"Vocab size: {len(vocab)}")
print(model)

## 2. Architecture explanation

**Pipeline:**
1. **Embedding** — token IDs → 96-dim dense vectors (shared for question & context)
2. **CNN** — 3 parallel Conv1d layers (k=2,3,4) with 96 filters each → detect bigrams, trigrams, 4-grams → projected to 128-dim
3. **BiLSTM** — 1-layer bidirectional LSTM (hidden=128) → 256-dim per token
4. **Attention** — scaled dot-product c2q + q2c fusion → 256-dim question-aware representations
5. **Start/End heads** — two Linear(256→1) layers predict logits over all context positions

## 3. Run inference on a sample question

In [ ]:
context  = "Photosynthesis is the process used by plants to convert light energy into chemical energy. It takes place inside organelles called chloroplasts."
question = "Where does photosynthesis take place?"

q_tokens = tokenize_with_offsets(question)
c_tokens = tokenize_with_offsets(context)

q_ids = torch.tensor([vocab.encode([t.text for t in q_tokens])])
c_ids = torch.tensor([vocab.encode([t.text for t in c_tokens])])

q_mask = torch.ones_like(q_ids, dtype=torch.bool)
c_mask = torch.ones_like(c_ids, dtype=torch.bool)

with torch.no_grad():
    start_logits, end_logits = model(q_ids, q_mask, c_ids, c_mask)

best_s, best_e = decode_best_span(start_logits[0], end_logits[0])
answer_tokens = [t.text for t in c_tokens[best_s:best_e+1]]
answer = context[c_tokens[best_s].start:c_tokens[best_e].end]

probs = torch.softmax(torch.stack([start_logits[0], end_logits[0]]), dim=-1)
confidence = (probs[0, best_s].item() * probs[1, best_e].item()) ** 0.5

print(f"Question : {question}")
print(f"Answer   : {answer}")
print(f"Confidence: {confidence:.4f}")
print(f"Span     : tokens [{best_s}, {best_e}], chars [{c_tokens[best_s].start}, {c_tokens[best_e].end}]")

## 4. Training history

Visualise the training curves from `saved_models/training_history.json`.

In [ ]:
import json
import matplotlib.pyplot as plt

with open("../saved_models/training_history.json") as f:
    history = json.load(f)

epochs = [h["epoch"] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, [h["train_loss"] for h in history], label="Train Loss")
ax1.plot(epochs, [h["val_loss"] for h in history], label="Val Loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Loss Curves"); ax1.legend()

ax2.plot(epochs, [h["train_f1"] for h in history], label="Train F1")
ax2.plot(epochs, [h["val_f1"] for h in history], label="Val F1")
ax2.plot(epochs, [h["train_em"] for h in history], label="Train EM", ls="--")
ax2.plot(epochs, [h["val_em"] for h in history], label="Val EM", ls="--")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Score"); ax2.set_title("Accuracy"); ax2.legend()

plt.tight_layout()
plt.savefig("../saved_models/training_history.png", dpi=150)
plt.show()